# 16 · Submission
Model: `14` (resnet18, random-slice training). Per knee: for each plane, the fluid-sensitive series with the most slices → same preprocessing as training → predict slices 6,8,…,18 → average → average the planes.

**Settings:** Internet OFF · GPU T4 · Inputs: competition data + `14_random_slice_15ep` notebook output.

In [ ]:
# Cell 1: setup
import os, glob, numpy as np, pandas as pd, torch, timm, pydicom, cv2
from sklearn.metrics import roc_auc_score

BASE = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['Sagittal', 'Coronal', 'Axial']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE, N_SLICES = 224, 24
TTA_SLICES = list(range(6, 19, 2))          # 6, 8, ..., 18

found = glob.glob('/kaggle/input/notebooks/*/*/resnet18_random_slice_15ep.pt')
print(DEVICE, found)
WEIGHTS = found[0]

In [ ]:
# Cell 2: load the trained model (offline: pretrained=False, weights come from the .pt)
model = timm.create_model('resnet18', pretrained=False, num_classes=12, in_chans=1)
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model = model.to(DEVICE).eval()
print('model loaded')

In [ ]:
# Cell 3: preprocessing, identical to training (02_preprocess)
def load_series(path):
    slices = [pydicom.dcmread(f) for f in glob.glob(f'{path}/*.dcm')]
    orient = np.array(slices[0].ImageOrientationPatient, float)
    normal = np.cross(orient[:3], orient[3:])
    slices.sort(key=lambda s: np.dot(normal, np.array(s.ImagePositionPatient, float)))
    return slices

def preprocess(path):
    vol = np.stack([cv2.resize(s.pixel_array.astype(np.float32), (IMG_SIZE, IMG_SIZE),
                               interpolation=cv2.INTER_AREA) for s in load_series(path)])
    vol = vol[np.linspace(0, len(vol) - 1, N_SLICES).round().astype(int)]
    lo, hi = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo) / (hi - lo + 1e-8), 0, 1)
    return (vol * 255).astype(np.uint8)                  # (24, 224, 224), same as the training .npy files

In [ ]:
# Cell 4: prediction for one volume and one knee
@torch.no_grad()
def predict_volume(vol):
    x = vol[TTA_SLICES].astype(np.float32) / 255.0
    x = (x - 0.45) / 0.225
    x = torch.from_numpy(x).unsqueeze(1).to(DEVICE)       # (7, 1, 224, 224)
    return torch.sigmoid(model(x)).mean(0).cpu().numpy()  # average over slices -> (12,)

def n_files(path):
    return len(glob.glob(f'{path}/*.dcm'))

def predict_study(study, series_df, split):
    meta = series_df[series_df['StudyInstanceUID'] == study]
    folder = f'{BASE}/{split}_series/{study}'
    preds = []
    for plane in PLANES:
        cands = meta[(meta['Anatomical_Plane'] == plane) & (meta['Fluid_Sensitive'] == 1)]
        if len(cands) == 0:
            continue
        path = max([f'{folder}/{s}' for s in cands['SeriesInstanceUID']], key=n_files)   # most slices, like training
        try:
            preds.append(predict_volume(preprocess(path)))
        except Exception as e:
            print('failed:', study, plane, e)
    if not preds:                                         # fallback: no usable fluid-sensitive series, try any series
        for s in meta['SeriesInstanceUID']:
            try:
                preds.append(predict_volume(preprocess(f'{folder}/{s}')))
                break
            except Exception as e:
                print('fallback failed:', study, s, e)
    return np.mean(preds, 0) if preds else None           # average over planes

In [ ]:
# Cell 5: end-to-end check on the labelled train knees (should be close to 14's ~0.78-0.79)
train = pd.read_csv(f'{BASE}/train.csv')
train_series = pd.read_csv(f'{BASE}/train_series.csv')
lab = train.dropna(subset=LABELS).reset_index(drop=True)

P = np.stack([predict_study(s, train_series, 'train') for s in lab['StudyInstanceUID']])
Y = lab[LABELS].values >= 0.5
aucs = [roc_auc_score(Y[:, i], P[:, i]) for i in range(len(LABELS)) if Y[:, i].min() != Y[:, i].max()]
print(f'{len(lab)} labelled knees, macro AUC = {np.mean(aucs):.4f}')

In [ ]:
# Cell 6: predict the test set and write submission.csv
sample = pd.read_csv(f'{BASE}/sample_submission.csv')
test_series = pd.read_csv(f'{BASE}/test_series.csv')

rows, n_fallback = [], 0
for i, study in enumerate(sample['StudyInstanceUID']):
    p = predict_study(study, test_series, 'test')
    if p is None:                                         # nothing usable at all -> neutral row, never a missing row
        p = np.full(len(LABELS), 0.5)
        n_fallback += 1
    rows.append([study, *p])
    if i % 100 == 0:
        print(i, '/', len(sample))

sub = pd.DataFrame(rows, columns=['StudyInstanceUID'] + LABELS)
assert list(sub.columns) == list(sample.columns), 'column mismatch'
assert len(sub) == len(sample) and sub['StudyInstanceUID'].equals(sample['StudyInstanceUID']), 'row mismatch'
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('fallback rows:', n_fallback)
sub.head()